# 041 · Jersey-number reader with labels from SportsMOT (E-BARD + our train clips, Google Colab)

**hoop-reid-tracker**, experiment 041 (same model and recipe as 033; only the training data changes).

### Why
033's reader, trained on NBA footage only (E-BARD), was right 94 % of the time on NBA test stubs but read a number on
only 32 % of the pieces of our validation clips, and those reads contradicted each other a quarter of the time: a change
of domain (other leagues, fonts, and SportsMOT's lower resolution and blur). 041 adds **in-domain labels**: one number
typed per ground-truth player of the 15 **train** clips of SportsMOT (`scripts/label_jersey_numbers.py`), turned into
16-crop stubs like E-BARD's (`scripts/export_jersey_stubs.py`): 1 FIBA game (women's Asia Cup 2019) and 2 NBA games at
SportsMOT's quality. The validation clips of SportsMOT are never used here: the reader is measured on them, on the
laptop, with the same gate as 033.

### Data
* E-BARD (CC BY 4.0), as in 033: train / val / test stubs;
* ours: `sportsmot_train_stubs.zip` in `MyDrive/hooptrack/`: `ours_train.txt` (12 clips) and `ours_val.txt` (the last clip of
  each game, held out). Our training stubs are repeated `OURS_REPEAT` times per epoch (they are ~10 % of the stubs).
* the confidence threshold is set on E-BARD val + our held-out clips together.


## 1 · GPU and libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers==5.21.0" huggingface_hub

## 2 · Configuration

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")
    WORK = Path("/content")
else:                                   # local smoke test
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

HF_DATASET = "GabrieleGiudici/E-BARD-JerseyNumberRecognition"
LOCAL_ZIP = os.environ.get("HOOPTRACK_ZIP")              # smoke test: a local zip instead of the download
MODEL = os.environ.get("HOOPTRACK_MODEL", "siglip")
HF_ID = os.environ.get("HOOPTRACK_HF_ID", "google/siglip-base-patch16-224")
TAG = f"jersey_reader_{MODEL}_041"
OURS_ZIP = DRIVE_DIR / "sportsmot_train_stubs.zip"    # from scripts/export_jersey_stubs.py
OURS_REPEAT = int(os.environ.get("HOOPTRACK_OURS_REPEAT", 3))
# Optional: start from the 033 reader (already trained on E-BARD) instead of the plain SigLIP tower, and train fewer
# epochs. Saves GPU time, but then 041 differs from 033 in two things (data AND starting point). Default: off.
INIT_FROM = None            # e.g. DRIVE_DIR / "runs" / "jersey_reader_siglip" / "jersey_reader_siglip"
if INIT_FROM is not None:
    EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 4))
RUN = (DRIVE_DIR / "runs" / TAG).resolve()

EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 10))
LR_TOWER, LR_HEAD, WEIGHT_DECAY, WARMUP = 1e-5, 1e-3, 1e-4, 1
BATCH = int(os.environ.get("HOOPTRACK_BATCH", 64))
TARGET_PRECISION = 0.95       # a piece is trusted when its confidence gives >= 95 % right answers (dataset's val)
SEED = 0
RUN.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print(f"experiment 041 | {MODEL} ({HF_ID}) | run {RUN}")

## 3 · The model code (exact copy of `src/hooptrack/reid/jersey_classifier.py`)
The laptop loads the trained model with the same file, so training and inference cannot drift apart.

In [ ]:
%%writefile jersey_classifier.py
"""Jersey-number reader trained on player crops, digit by digit (experiment 033).

Why digit by digit
------------------
The training data (E-BARD, NBA 2024-25 broadcast) has only 52 different numbers, and our val clips (NCAA,
European leagues) have others. A classifier with one output per number can never answer a number it did not see
in training. Jersey-number papers therefore read the number the way a person does, one digit at a time:

* ``length``: how many digits are visible: 0 (no number readable: back turned, arm in front, too small),
  1 or 2;
* ``tens``: the first digit of a two-digit number (0-9; "00" is tens 0, units 0);
* ``units``: the last digit (0-9), shared by one- and two-digit numbers.

The probability of a number is the product of its parts::

    P(no number) = P(length = 0)
    P("7")       = P(length = 1) * P(units = 7)
    P("23")      = P(length = 2) * P(tens = 2) * P(units = 3)

so the 111 answers (no number, 0-9, 00-99) are scored from 3 small heads, and every digit is learned from every
number that contains it.

The model
---------
The image tower of a pretrained model (default: SigLIP base, web image-text pretraining, so it has seen a lot of
printed text and digits; the 028 ReID model is the same tower) on the player crop at 256 x 128 (height x width,
not deformed, position vectors interpolated as in ``hooptrack.reid.siglip``), then the three linear heads.

Reading a piece
---------------
One frame is often not enough (the number turns away, motion blur). ``read_piece`` averages the 111-way
distributions of the crops of a piece and answers the most probable number with its mean probability as
confidence; the caller decides the confidence needed (calibrated on the training data's val split).

This file is used by the Colab notebook (training: it writes an exact copy) and by the laptop (inference), so the
two cannot drift apart.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F  # noqa: N812 (PyTorch convention)
from torch import nn

INPUT_SIZE = (256, 128)                      # (height, width), as the ReID crops
NUMBERS = ["nan"] + [str(d) for d in range(10)] + [f"{t}{d}" for t in range(10) for d in range(10)]   # 111 answers


def encode(label: str) -> tuple[int, int, int]:
    """'nan' / '7' / '23' / '00' -> (length, tens, units); -100 = no target for that head (ignored by the loss)."""
    label = str(label).strip()
    if label.lower() == "nan" or not label.isdigit() or len(label) > 2:
        return 0, -100, -100
    if len(label) == 1:
        return 1, -100, int(label)
    return 2, int(label[0]), int(label[1])


def number_log_probs(length_logits: torch.Tensor, tens_logits: torch.Tensor,
                     units_logits: torch.Tensor) -> torch.Tensor:
    """(B, 3), (B, 10), (B, 10) logits -> (B, 111) log-probabilities over ``NUMBERS``."""
    ll, lt, lu = (F.log_softmax(x.float(), dim=1) for x in (length_logits, tens_logits, units_logits))
    none = ll[:, :1]
    one = ll[:, 1:2] + lu                                            # (B, 10)
    two = (ll[:, 2:3, None] + lt[:, :, None] + lu[:, None, :]).flatten(1)   # (B, 100), tens-major
    return torch.cat([none, one, two], dim=1)


class JerseyNumberModel(nn.Module):
    """Pretrained image tower + 3 digit heads."""

    def __init__(self, tower, dim: int, kind: str = "siglip") -> None:
        super().__init__()
        self.tower, self.kind = tower, kind
        self.length = nn.Linear(dim, 3)
        self.tens = nn.Linear(dim, 10)
        self.units = nn.Linear(dim, 10)

    def features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out = self.tower(pixel_values=pixel_values, interpolate_pos_encoding=True)
        return out.pooler_output if self.kind == "siglip" else out.image_embeds

    def forward(self, pixel_values: torch.Tensor):
        f = self.features(pixel_values)
        return self.length(f), self.tens(f), self.units(f)


def loss_fn(logits, targets) -> torch.Tensor:
    """Cross-entropy of each head; a head with no target for a sample (-100) does not count."""
    (ll, lt, lu), (yl, yt, yu) = logits, targets
    loss = F.cross_entropy(ll.float(), yl)
    if (yt >= 0).any():
        loss = loss + F.cross_entropy(lt.float(), yt, ignore_index=-100)
    if (yu >= 0).any():
        loss = loss + F.cross_entropy(lu.float(), yu, ignore_index=-100)
    return loss


def load_tower(path: str, kind: str):
    from transformers import CLIPVisionModelWithProjection, SiglipVisionModel

    if kind == "siglip":
        tower = SiglipVisionModel.from_pretrained(path)
        return tower, tower.config.hidden_size
    tower = CLIPVisionModelWithProjection.from_pretrained(path)
    return tower, tower.config.projection_dim


def save(model: JerseyNumberModel, processor, out_dir: str | Path, meta: dict) -> None:
    """Folder = the tower (save_pretrained) + its processor (mean / std) + heads.pt + jersey_reader.json."""
    out_dir = Path(out_dir)
    model.tower.save_pretrained(out_dir)
    processor.save_pretrained(out_dir)
    torch.save({k: v for k, v in model.state_dict().items() if not k.startswith("tower.")}, out_dir / "heads.pt")
    (out_dir / "jersey_reader.json").write_text(json.dumps({"kind": model.kind, "input_size": list(INPUT_SIZE),
                                                            **meta}, indent=2))


def test_transform(input_size: tuple[int, int], mean, std):
    """Resize to (h, w), to tensor, normalise: identical to ``hooptrack.reid.siglip.reid_transform`` (copied so that
    this file also works on its own, in the Colab notebook)."""
    import torchvision.transforms as T  # noqa: N812 (torchvision convention)

    return T.Compose([T.Resize(tuple(input_size)), T.ToTensor(), T.Normalize(mean, std)])


def load(path: str | Path, device: str = "cpu"):
    """-> (model in eval mode, test transform, meta)."""
    from transformers import AutoImageProcessor

    path = Path(path)
    meta = json.loads((path / "jersey_reader.json").read_text())
    tower, dim = load_tower(str(path), meta["kind"])
    model = JerseyNumberModel(tower, dim, meta["kind"])
    missing, unexpected = model.load_state_dict(torch.load(path / "heads.pt", map_location="cpu"), strict=False)
    if [k for k in missing if not k.startswith("tower.")] or unexpected:
        raise ValueError(f"heads.pt does not match the model: {missing} {unexpected}")
    proc = AutoImageProcessor.from_pretrained(str(path))
    return model.eval().to(device), test_transform(tuple(meta["input_size"]), proc.image_mean, proc.image_std), meta


def read_piece(probs: np.ndarray) -> tuple[str, float]:
    """(n_crops, 111) probabilities of the crops of one piece -> (number or 'nan', confidence)."""
    mean = np.asarray(probs, float).mean(axis=0)
    k = int(mean.argmax())
    return NUMBERS[k], float(mean[k])

## 4 · Download and index the stubs
`train.txt` / `val.txt` / `test.txt`: one line per stub, `annotated/<game>_<play>-track_<id> 16 <label>`. The 16 crops
of a stub are the PNG files of its folder.

In [ ]:
import zipfile, random
import numpy as np, pandas as pd

DATA = WORK / "ebard_jnr"
if not (DATA / "train.txt").is_file():
    if LOCAL_ZIP:
        zpath = Path(LOCAL_ZIP)
    else:
        from huggingface_hub import hf_hub_download
        zpath = Path(hf_hub_download(HF_DATASET, "all.zip", repo_type="dataset", local_dir=str(WORK / "ebard_dl")))
    with zipfile.ZipFile(zpath) as z:
        members = [n for n in z.namelist() if n.startswith("annotated/") or n.endswith(".txt") or n.endswith(".csv")]
        z.extractall(DATA, members)

def stubs(split: str) -> pd.DataFrame:
    rows = []
    for line in (DATA / f"{split}.txt").read_text().splitlines():
        parts = line.split()
        if len(parts) < 3:
            continue
        folder, label = parts[0], parts[-1]
        frames = sorted((DATA / folder).glob("*.png"), key=lambda p: int(p.stem.split("_")[-1]))
        if frames:
            rows.append({"stub": folder, "label": label, "frames": [str(p) for p in frames[:16]]})
    return pd.DataFrame(rows)

S = {s: stubs(s) for s in ("train", "val", "test")}
for s, df in S.items():
    print(f"{s}: {len(df)} stubs, {sum(map(len, df.frames))} crops, {df.label.nunique()} labels, "
          f"{(df.label == 'nan').mean():.1%} 'nan'")

### Our stubs (SportsMOT train clips, labelled by hand)


In [ ]:
OURS = WORK / "sportsmot_stubs"
if not (OURS / "ours_train.txt").is_file():
    with zipfile.ZipFile(OURS_ZIP) as z:
        z.extractall(OURS)

def our_stubs(name: str) -> pd.DataFrame:
    rows = []
    for line in (OURS / name).read_text().splitlines():
        parts = line.split()
        if len(parts) < 3:
            continue
        folder, label = parts[0], parts[-1]
        frames = sorted((OURS / folder).glob("*.jpg"), key=lambda p: int(p.stem.split("_")[-1]))
        if frames:
            rows.append({"stub": folder, "label": label, "frames": [str(p) for p in frames[:16]]})
    return pd.DataFrame(rows)

S["ours_train"], S["ours_val"] = our_stubs("ours_train.txt"), our_stubs("ours_val.txt")
S["ebard_val"] = S["val"]
S["train"] = pd.concat([S["train"]] + [S["ours_train"]] * OURS_REPEAT, ignore_index=True)
S["val"] = pd.concat([S["ebard_val"], S["ours_val"]], ignore_index=True)    # best epoch and threshold on both
for s in ("ours_train", "ours_val", "train", "val"):
    df = S[s]
    print(f"{s}: {len(df)} stubs, {df.label.nunique()} labels, {(df.label == 'nan').mean():.1%} 'nan'")


### What the reader sees
Four stubs per row: a few of their 16 crops, with the label. The number is often small, turned or hidden.

In [ ]:
import cv2
import matplotlib.pyplot as plt

random.seed(SEED)
fig, axes = plt.subplots(3, 8, figsize=(16, 9))
for r, (_, row) in enumerate(S["train"].sample(3, random_state=SEED).iterrows()):
    for c, f in enumerate(row.frames[::2][:8]):
        axes[r, c].imshow(cv2.imread(f)[..., ::-1]); axes[r, c].axis("off")
    axes[r, 0].set_title(f"label {row.label}", fontsize=10, loc="left")
plt.tight_layout(); plt.show()

## 5 · Data pipeline
Every crop of a training stub is one training sample with the stub's label (a few crops of a stub may not show the
number well: that noise is accepted, as in the paper). Test transform = `test_transform` of the module, the laptop's.

In [ ]:
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import jersey_classifier as jc

torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from transformers import AutoImageProcessor
processor = AutoImageProcessor.from_pretrained(HF_ID)
MEAN, STD = processor.image_mean, processor.image_std
H, W = jc.INPUT_SIZE

train_tf = T.Compose([
    T.Resize((H, W)),
    T.RandomApply([T.RandomResizedCrop((H, W), scale=(0.8, 1.0), ratio=(0.45, 0.55))], p=0.5),   # shifts / zoom
    T.RandomApply([T.Resize((H // 3, W // 3)), T.Resize((H, W))], p=0.3),        # small, low-resolution players
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),                 # motion / focus blur
    T.ColorJitter(brightness=0.3, contrast=0.3),                                 # no hue change: kit colours are real
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.1), value="random"),                  # an arm in front of the number
])
test_tf = jc.test_transform(jc.INPUT_SIZE, MEAN, STD)

class Crops(Dataset):
    def __init__(self, df, tf):
        self.items = [(f, jc.encode(lbl)) for f_list, lbl in zip(df.frames, df.label) for f in f_list]
        self.tf = tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        f, (l, t, u) = self.items[i]
        return self.tf(Image.open(f).convert("RGB")), torch.tensor([l, t, u])

WORKERS = 2 if IN_COLAB else 0
train_loader = DataLoader(Crops(S["train"], train_tf), batch_size=BATCH, shuffle=True, num_workers=WORKERS,
                          drop_last=True, pin_memory=DEVICE == "cuda")
print(f"{len(train_loader.dataset)} training crops, {len(train_loader)} batches per epoch, device {DEVICE}")

## 6 · Reading stubs (the evaluation)
Each crop gives 111 probabilities; a stub's answer is the most probable number of their mean, with that mean
probability as confidence (`jc.read_piece`, what the laptop does with a piece of a track).

In [ ]:
@torch.inference_mode()
def read_stubs(model, df, batch=128) -> pd.DataFrame:
    model.eval()
    files = [f for fl in df.frames for f in fl]
    probs = []
    for i in range(0, len(files), batch):
        x = torch.stack([test_tf(Image.open(f).convert("RGB")) for f in files[i:i + batch]]).to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits = model(x)
        probs.append(jc.number_log_probs(*logits).exp().cpu().numpy())
    probs = np.concatenate(probs)
    out, k = [], 0
    for _, row in df.iterrows():
        n = len(row.frames)
        num, conf = jc.read_piece(probs[k:k + n]); k += n
        out.append({"stub": row.stub, "label": row.label, "pred": num, "conf": conf})
    r = pd.DataFrame(out)
    r["right"] = r.pred == r.label
    return r

def summary(r: pd.DataFrame, min_conf: float = 0.0) -> dict:
    ans = r[(r.conf >= min_conf) & (r.pred != "nan")]          # answered with a number
    numbered = r[r.label != "nan"]
    return {"stub_accuracy": round(r.right.mean(), 3),
            "answered_share": round(len(ans) / len(r), 3),
            "precision_when_answered": round(ans.right.mean(), 3) if len(ans) else None,
            "numbers_found": round(((numbered.conf >= min_conf) & numbered.right).mean(), 3) if len(numbered) else None}

## 7 · Training

In [ ]:
import math, time

if INIT_FROM is not None:                      # continue from the 033 reader
    model, _, _ = jc.load(INIT_FROM, DEVICE)
    model.train()
    print(f"starting from {INIT_FROM}")
else:
    tower, DIM = jc.load_tower(HF_ID, MODEL)
    model = jc.JerseyNumberModel(tower, DIM, MODEL).to(DEVICE)
heads = [p for n, p in model.named_parameters() if not n.startswith("tower.")]
opt = torch.optim.AdamW([{"params": model.tower.parameters(), "lr": LR_TOWER},
                         {"params": heads, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
TOTAL = EPOCHS * len(train_loader)
def lr_factor(step):
    w = WARMUP * len(train_loader)
    return (step + 1) / w if step < w else 0.5 * (1 + math.cos(math.pi * (step - w) / max(1, TOTAL - w)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

start, history, best = 0, [], -1.0
ckpt = RUN / "last.pth"
if ckpt.is_file():
    s = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
    scaler.load_state_dict(s["scaler"]); start, history, best = s["epoch"] + 1, s["history"], s["best"]
    print(f"resuming at epoch {start + 1}")

OUT = RUN / TAG                                        # the best model, in the folder the laptop loads
for epoch in range(start, EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = jc.loss_fn(model(x), (y[:, 0], y[:, 1], y[:, 2]))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    val = summary(read_stubs(model, S["val"]))
    history.append({"epoch": epoch + 1, "loss": tot / len(train_loader), **val})
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss {tot / len(train_loader):.3f}  val {val}  ({time.time() - t0:.0f} s)")
    if val["stub_accuracy"] > best:                    # keep the best epoch on the DATASET's val split
        best = val["stub_accuracy"]
        jc.save(model, processor, OUT, {"model": MODEL, "hf_id": HF_ID, "epoch": epoch + 1})
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "history": history, "best": best}, ckpt)
print("best val stub accuracy", best, "->", OUT)

## 8 · The confidence to trust a piece, and the test
The saved folder is re-loaded with `jc.load` (what the laptop does). On the dataset's val split: the lowest
confidence at which 95 % of the stubs answered with a number are right. Then the test split at that confidence.

In [ ]:
import json
model, test_tf, meta = jc.load(OUT, DEVICE)
rv, rt = read_stubs(model, S["val"]), read_stubs(model, S["test"])
ro = read_stubs(model, S["ours_val"])
curve = []
for c in np.round(np.arange(0.0, 1.0001, 0.02), 2):
    curve.append({"min_conf": float(c), **summary(rv, c)})
ok = [r for r in curve if r["precision_when_answered"] is not None and r["precision_when_answered"] >= TARGET_PRECISION]
MIN_CONF = ok[0]["min_conf"] if ok else 1.0
print(f"min_conf {MIN_CONF} (first confidence with >= {TARGET_PRECISION:.0%} right on the dataset's val)")
results = {"val": summary(rv, MIN_CONF), "test": summary(rt, MIN_CONF), "ours_val": summary(ro, MIN_CONF),
           "ours_val_all_answers": summary(ro),
           "val_all_answers": summary(rv), "test_all_answers": summary(rt)}
print(json.dumps(results, indent=2))
meta.update({"min_conf": MIN_CONF, "target_precision": TARGET_PRECISION})
(OUT / "jersey_reader.json").write_text(json.dumps(meta, indent=2))

plt.figure(figsize=(6, 4))
cv_ = pd.DataFrame(curve)
plt.plot(cv_.min_conf, cv_.precision_when_answered, label="right when answered")
plt.plot(cv_.min_conf, cv_.answered_share, label="share of stubs answered")
plt.axvline(MIN_CONF, color="k", ls="--", lw=1); plt.xlabel("confidence needed"); plt.legend(); plt.grid(alpha=0.3)
plt.title("E-BARD val + our held-out clips"); plt.savefig(RUN / "confidence_curve.png", dpi=80); plt.show()

## 9 · Save the report and take the model home
1. Run the next cell: it writes `eval_041.json` and zips the model folder.
2. Download from `MyDrive/hooptrack/runs/jersey_reader_siglip_041/`: `jersey_reader_siglip_041.zip`, `eval_041.json`,
   `confidence_curve.png`. Unzip the model into `weights/jersey_reader_siglip_041/` and put the report in
   `experiments/041_jersey_reader_sportsmot/`.
3. Next, on the laptop: read the pieces of our 15 val clips and measure the coverage (the gate of 041, same as 033).

In [ ]:
import shutil
report = {"experiment": "041", "dataset": HF_DATASET, "licence": "CC BY 4.0", "model": MODEL, "hf_id": HF_ID,
          "recipe": {"epochs": EPOCHS, "lr_tower": LR_TOWER, "lr_head": LR_HEAD, "weight_decay": WEIGHT_DECAY,
                     "warmup_epochs": WARMUP, "batch": BATCH, "input_size": list(jc.INPUT_SIZE), "flip": False},
          "data": {s: {"stubs": len(df), "crops": int(sum(map(len, df.frames)))} for s, df in S.items()}, "ours_repeat": OURS_REPEAT, "init_from": str(INIT_FROM),
          "min_conf": MIN_CONF, "results": results, "curve_val": curve, "history": history}
(RUN / "eval_041.json").write_text(json.dumps(report, indent=2))
shutil.make_archive(str(RUN / TAG), "zip", root_dir=RUN, base_dir=TAG)
print("wrote", RUN / "eval_041.json", "and", RUN / f"{TAG}.zip")